# تجربة واحدة للنموذج الحقيقي — SmartGrade

يستخدم هذا الدفتر محرك التصحيح نفسه في المشروع وإجابة العلوم الاصطناعية ذات المرجع17/30.
لا يحتاج حساب مؤسسة، ولا يغير بياناتك أو يعتمد درجات. لا يستدعي المزود حتى تفعّل APPROVE_ONE_REQUEST.
المفتاح يُقرأ من Colab Secrets فقط، ولا يُطبع أو يُحفظ في التقرير. أدخل اسم النموذج إن لم تحفظه ضمن Secrets.
يشغل طلبًا واحدًا دون إعادة محاولة تلقائية؛ النتيجة مقارنة لعينة واحدة وليست نسبة دقة عامة.


In [ ]:
# @title 1. تجهيز نسخة معزولة من محرك التصحيح
import base64, hashlib, json, subprocess, sys
from pathlib import Path
SOURCES_B64 = "..."
source_files = json.loads(base64.b64decode(SOURCES_B64).decode("utf-8"))
MODEL_TEST_DIR = Path("/content/SmartGrade-One-Model-Test-de39deb7")
MODEL_TEST_DIR.mkdir(parents=True, exist_ok=True)
if MODEL_TEST_DIR.is_symlink():
    raise RuntimeError("اختر مجلدًا غير رمزي")
for name, entry in source_files.items():
    destination = MODEL_TEST_DIR / name
    if MODEL_TEST_DIR.resolve() not in destination.resolve().parents:
        raise ValueError("مسار غير آمن")
    raw = entry["text"].encode("utf-8")
    if entry["sha"] and hashlib.sha1(b"blob " + str(len(raw)).encode() + b"\0" + raw).hexdigest() != entry["sha"]:
        raise ValueError("بصمة المصدر غير صحيحة")
    if destination.exists() and destination.read_bytes() != raw:
        raise FileExistsError("ملف معدل موجود؛ اختر مجلدًا جديدًا لحفظ تعديلاتك")
    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_bytes(raw)
MODEL_TEST_VENV = Path("/content/smartgrade-one-model-venv")
MODEL_TEST_PYTHON = MODEL_TEST_VENV / "bin/python"
if not MODEL_TEST_PYTHON.exists():
    subprocess.run([sys.executable, "-m", "venv", str(MODEL_TEST_VENV)], check=True)
subprocess.run([str(MODEL_TEST_PYTHON), "-m", "pip", "install", "--quiet",
                "httpx==0.28.1", "pydantic==2.14.3"], check=True)
print("محرك التصحيح جاهز. لم يُرسل أي طلب إلى النموذج.")


In [ ]:
# @title 2. طلب واحد باستخدام Colab Secrets — فعّله عند الموافقة على تكلفة الطلب
MODEL_NAME = "" # @param {type:"string"}
API_ROOT = "" # @param {type:"string"}
APPROVE_ONE_REQUEST = False # @param {type:"boolean"}
def run_one_model_from_secrets():
    import os
    from google.colab import userdata
    def read_secret(names):
        for name in names:
            try:
                value = userdata.get(name)
                if value and value.strip():
                    return value.strip()
            except Exception:
                pass
        return ""
    key = read_secret(["SMARTGRADE_LLM_API_KEY", "LLM_API_KEY"])
    model = MODEL_NAME.strip() or read_secret(["SMARTGRADE_LLM_MODEL", "LLM_MODEL"])
    base = API_ROOT.strip() or read_secret(["SMARTGRADE_LLM_BASE_URL", "LLM_BASE_URL"]) or "https://api.openai.com/v1"
    if not key:
        raise RuntimeError("لم يتوفر المفتاح: تحقق من اسم SMARTGRADE_LLM_API_KEY أو LLM_API_KEY ومن السماح لهذا الدفتر بقراءته.")
    if not model:
        raise RuntimeError("أدخل اسم النموذج في MODEL_NAME أو خزنه باسم SMARTGRADE_LLM_MODEL أو LLM_MODEL.")
    child_env = {**os.environ, "LLM_API_KEY": key, "LLM_MODEL": model, "LLM_BASE_URL": base,
                 "ALLOW_EXTERNAL_AI": "true", "ALLOW_PAID_API": "true"}
    try:
        result = subprocess.run([str(MODEL_TEST_PYTHON), str(MODEL_TEST_DIR / "one_model_check.py")],
                                cwd=MODEL_TEST_DIR, env=child_env, capture_output=True, text=True, timeout=75)
        if result.stdout:
            print(result.stdout)
        if result.returncode:
            print("لم يكتمل تشغيل محرك الاختبار؛ لم تُعرض سجلات تحتوي على إعدادات حساسة.")
    finally:
        del key, child_env
if APPROVE_ONE_REQUEST:
    run_one_model_from_secrets()
else:
    print("لم يُستدعَ النموذج. فعّل APPROVE_ONE_REQUEST ثم شغّل هذه الخلية للموافقة على طلب واحد.")


In [ ]:
# @title 3. عرض آخر نتيجة ومقارنتها بالمرجع
from IPython.display import display, Markdown
RESULT_FILE = MODEL_TEST_DIR / "single-model-result.json"
if RESULT_FILE.exists():
    result = json.loads(RESULT_FILE.read_text())
    print("آخر نتيجة محفوظة — وقت UTC:", result["timestamp_utc"])
    if result.get("model_evaluated"):
        comparison = result["comparison"]
        print("اقتراح النموذج:", comparison["proposed_total"], "/30 | المرجع:", comparison["reference_total"], "/30")
        table = "| السؤال | المرجع | اقتراح النموذج | الفارق المطلق |\n|---|---:|---:|---:|\n"
        for row in comparison["questions"]:
            table += f"| {row['number']} | {row['reference']} | {row['proposed']} | {row['absolute_error']} |\n"
        display(Markdown(table))
        print("المجموع وحده لا يثبت الجودة؛ راجع تفاصيل المعايير في ملف النتيجة.")
    else:
        print(result.get("message", result.get("exception_type", "لم تكتمل التجربة")))
        print("رمز التشخيص:", result.get("code", "غير متاح"))
else:
    print("لا توجد نتيجة بعد. شغّل خلية الطلب عند جاهزية المفتاح واسم النموذج.")
